https://www.kimi.com/chat/d41m8ne0ftllr9osecig

In [1]:
!pip install tiktoken pypdf aiohttp aiofiles

In [ ]:
#!/usr/bin/env python3
"""
PDF → Markdown pipeline optimised for RAG.
-Splits PDFs into page-chunks
-Calls a local LM-Studio /chat/completions endpoint
-Returns clean, citation-ready Markdown with page anchors, LaTeX math, etc.
"""

import os
import tempfile
import asyncio
from pathlib import Path
from typing import List, Tuple
from dataclasses import dataclass
import aiohttp
import json
import random
from enum import Enum

try:
    from pypdf import PdfReader, PdfWriter
except ImportError:
    raise SystemExit("Please:  pip install pypdf")

try:
    import aiofiles
except ImportError:
    raise SystemExit("Please:  pip install aiofiles")

# ------------------------------------------------------------------
# CONFIGURATION CONSTANTS
# ------------------------------------------------------------------
class ModelConfig(Enum):
    LOCAL_DEFAULT = "qwen/qwen3-coder-30b"   # <-- your model id here

MODEL = ModelConfig.LOCAL_DEFAULT.value
FOLDER_PATH      = "pdfs_20_2"        # folder containing PDFs
OUTPUT_DIR       = f"lmstudio_output_md/{MODEL.replace('/', '_')}"
PAGES_PER_CHUNK  = 1                  # set higher if your context allows
TEMPERATURE      = 0.0
MAX_CONCURRENT   = 1                  # LM-Studio parallelism

LMSTUDIO_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://192.168.10.60:12345/v1").rstrip("/")
LMSTUDIO_API_KEY  = os.getenv("LMSTUDIO_API_KEY", "")

# Token budget knobs
CONTEXT_WINDOW_TOKENS = int(os.getenv("CTX_TOKENS", "8192"))
MAX_OUTPUT_TOKENS     = int(os.getenv("MAX_OUTPUT_TOKENS", "1024"))
TOKEN_SAFETY_MARGIN   = int(os.getenv("TOKEN_SAFETY_MARGIN", "200"))
MAX_CHARS_PER_REQUEST = 15000         # crude guard even after token counting
# ------------------------------------------------------------------

@dataclass
class ChunkInfo:
    pdf_name: str
    chunk_index: int
    chunk_path: str
    page_range_label: str


class TokenCounter:
    """Tiktoken if available, else 4-char ≈ 1 token heuristic."""
    def __init__(self):
        self._enc = None
        self._mode = "heuristic"
        try:
            import tiktoken
            self._enc = tiktoken.get_encoding("cl100k_base")
            self._mode = "tiktoken:cl100k_base"
        except Exception:
            pass

    def count(self, text: str) -> int:
        if not text:
            return 0
        if self._enc:
            try:
                return len(self._enc.encode(text))
            except Exception:
                pass
        return max(1, len(text) // 4)

    def __repr__(self) -> str:
        return f"<TokenCounter mode={self._mode}>"


# ------------------------------------------------------------------
# SYSTEM PROMPT  –  RAG-OPTIMISED
# ------------------------------------------------------------------
SYSTEM_PROMPT = """\
You are a “PDF → Markdown” converter optimised for downstream Retrieval-Augmented-Generation (RAG).

OUTPUT RULES
1.  Return ONLY the Markdown text—no wrapper sentences, no “Here is the result…” preamble.
2.  Preserve the *logical* structure: title → sections → sub-sections.  
    Use ATX headings (`#`…`######`) exactly as they appear in the document; do **not** invent new ones.
3.  Keep the original page boundaries visible as:
    `<!-- pg N -->`
    Place the comment on its own line *before* the first heading or paragraph that starts on page N.
4.  If a paragraph is split across pages, insert the comment only once, at the start of the split.
5.  Convert every table to a GitHub-flavoured Markdown table.  
    - Add a caption line immediately under the table in *italic* if the original table had a caption/title.  
    - If cells contain multi-line text, replace line breaks inside cells with `<br>`.
6.  Inline math:  `$ E = mc^2 $`  
    Display math:  `$$ \int_{-\infty}^{\infty} e^{-x^2} dx = \sqrt{\pi} $$`  
    (Use LaTeX syntax even if the PDF uses Unicode or images.)
7.  Keep all numbered / bulleted lists exactly as numbered; do **not** restart counters unless the source does.
8.  Footnotes: place them *inline* at the end of the sentence in which the reference occurs, using `^N:` syntax:
    `…some claim^1: This is the footnote text.`
9.  Remove running headers, footers, page numbers that appear as artefacts—**unless** they are part of the
    genuine content (e.g. chapter title in a header).
10. If a figure or image is referenced in the text, reproduce the exact caption as a blockquote:
    `> **Figure 3:** Schematic of the proposed pipeline.`
    If no caption exists, omit the figure entirely (do not hallucinate).
11. Retain hyperlinks (copy the visible text plus URL).  
    If the URL is broken or truncated, keep the visible text and add `(link broken)` after it.
12. Do **not** summarise or skip “boring” sections—emit every recoverable sentence.
13. Finish with a single blank line; no trailing XML or JSON.
"""
# ------------------------------------------------------------------


class AsyncPDFProcessor:
    def __init__(self, base_url: str, api_key: str = ""):
        self.base_url = base_url
        self.api_key  = api_key or None
        self.tok      = TokenCounter()

    # ---------- FS & splitting ----------
    def get_pdf_files(self, folder_path: str, f_limit: int = 9999) -> List[Path]:
        folder = Path(folder_path)
        if not folder.exists():
            raise SystemExit(f"Folder {folder_path} does not exist.")
        pdf_files = list(folder.glob("*.pdf"))[:f_limit]
        if not pdf_files:
            raise SystemExit(f"No PDF files found in {folder_path}")
        return pdf_files

    def split_pdf_to_chunks(self, pdf_path: str, pages_per_chunk: int) -> List[ChunkInfo]:
        reader = PdfReader(pdf_path)
        n_pages = len(reader.pages)
        chunks: List[ChunkInfo] = []
        pdf_name = Path(pdf_path).stem

        for chunk_idx, start in enumerate(range(0, n_pages, pages_per_chunk)):
            end = min(start + pages_per_chunk, n_pages)
            writer = PdfWriter()
            for i in range(start, end):
                writer.add_page(reader.pages[i])

            fd, tmp_pdf = tempfile.mkstemp(
                suffix=f".{pdf_name}.chunk{chunk_idx:03d}.{start+1}-{end}.pdf"
            )
            os.close(fd)
            with open(tmp_pdf, "wb") as f:
                writer.write(f)

            page_range_label = f"pages {start+1}–{end}"
            chunks.append(ChunkInfo(
                pdf_name=pdf_name,
                chunk_index=chunk_idx,
                chunk_path=tmp_pdf,
                page_range_label=page_range_label
            ))
        return chunks

    # ---------- helpers ----------
    def extract_text_from_pdf(self, pdf_path: str) -> str:
        try:
            reader = PdfReader(pdf_path)
            parts = []
            for i, page in enumerate(reader.pages, start=1):
                try:
                    txt = page.extract_text() or ""
                except Exception:
                    txt = ""
                if txt.strip():
                    parts.append(f"\n\n[Page {i}]\n{txt.strip()}")
                else:
                    parts.append(f"\n\n[Page {i}]")
            return "".join(parts)
        except Exception as e:
            return f"[EXTRACTION_ERROR] {e}"

    def clamp(self, s: str, max_chars: int) -> str:
        return s if len(s) <= max_chars else s[:max_chars]

    async def _with_retries(self, coro_factory, *, tries=5, base=0.7, max_sleep=10.0):
        last = None
        for i in range(tries):
            try:
                return await coro_factory()
            except Exception as e:
                last = e
                msg = str(e).lower()
                retryable = any(s in msg for s in ("429", "timeout", "temporarily", "5", "rate", "connection", "reset", "refused"))
                if retryable and i < tries - 1:
                    sleep_for = min(max_sleep, base * (2 ** i)) + random.random()
                    await asyncio.sleep(sleep_for)
                else:
                    break
        raise last

    # ---------- LM Studio chat/completions ----------
    async def chat_markdownify(self, session: aiohttp.ClientSession, model: str,
                               system: str, user: str, temperature: float, max_tokens: int) -> str:
        headers = {"Content-Type": "application/json"}
        if self.api_key:
            headers["Authorization"] = f"Bearer {self.api_key}"

        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            "temperature": temperature,
            "max_tokens": max_tokens,
            "stream": False
        }
        url = f"{self.base_url}/chat/completions"
        async with session.post(url, json=payload, headers=headers) as r:
            if r.status != 200:
                t = await r.text()
                raise Exception(f"LM Studio API error {r.status}: {t}")
            data = await r.json()
            return data["choices"][0]["message"]["content"].strip()

    # ---------- per-chunk pipeline ----------
    async def process_one_chunk(
        self,
        session: aiohttp.ClientSession,
        semaphore: asyncio.Semaphore,
        chunk: ChunkInfo
    ) -> Tuple[int, str]:
        async with semaphore:
            try:
                raw_text = self.extract_text_from_pdf(chunk.chunk_path)
                clamped = self.clamp(raw_text, MAX_CHARS_PER_REQUEST)

                user_prompt = (
                    f"Convert this extracted PDF chunk ({chunk.page_range_label}) to Markdown.\n\n"
                    f"--- BEGIN EXTRACTED TEXT ---\n{clamped}\n--- END EXTRACTED TEXT ---"
                )

                sys_toks = self.tok.count(SYSTEM_PROMPT)
                usr_toks = self.tok.count(user_prompt)
                in_toks  = sys_toks + usr_toks
                available = max(256, CONTEXT_WINDOW_TOKENS - in_toks - TOKEN_SAFETY_MARGIN)
                max_out = max(64, min(MAX_OUTPUT_TOKENS, available))

                print(
                    f"🔢 {chunk.pdf_name} | {chunk.page_range_label} | "
                    f"tokens: in={in_toks} (sys={sys_toks}, user={usr_toks}) | "
                    f"max_out={max_out} | ctx={CONTEXT_WINDOW_TOKENS} | {self.tok}"
                )

                md = await self._with_retries(
                    lambda: self.chat_markdownify(
                        session,
                        MODEL,
                        SYSTEM_PROMPT,
                        user_prompt,
                        temperature=TEMPERATURE,
                        max_tokens=int(max_out)
                    )
                )
                header = (
                    f"*Token usage:* input={in_toks} (sys={sys_toks}, user={usr_toks}), "
                    f"max_out={max_out}, ctx={CONTEXT_WINDOW_TOKENS}\n"
                )
                formatted = f"\n\n---\n*Chunk {chunk.page_range_label}*\n{header}---\n\n{md.strip()}\n"
                print(f"✅ {chunk.pdf_name} | {chunk.page_range_label}")
                return chunk.chunk_index, formatted
            except Exception as e:
                print(f"⚠️ {chunk.pdf_name} | {chunk.page_range_label} failed: {e}")
                return chunk.chunk_index, f"\n\n---\n*Error processing {chunk.page_range_label}: {e}*\n---\n\n"
            finally:
                try:
                    os.remove(chunk.chunk_path)
                except OSError:
                    pass

    async def process_single_pdf(self, session: aiohttp.ClientSession, pdf_path: str, output_dir: str):
        pdf_name = Path(pdf_path).stem
        print(f"\n📄 Processing PDF: {Path(pdf_path).name}")
        chunks = self.split_pdf_to_chunks(pdf_path, PAGES_PER_CHUNK)
        print(f"🔧 Split into {len(chunks)} chunks (window={MAX_CONCURRENT})")

        semaphore = asyncio.Semaphore(MAX_CONCURRENT)
        tasks = [asyncio.create_task(self.process_one_chunk(session, semaphore, ch)) for ch in chunks]
        results: List[Tuple[int, str]] = await asyncio.gather(*tasks)

        results.sort(key=lambda x: x[0])
        merged_md = "".join(md for _, md in results).strip() + "\n"

        os.makedirs(output_dir, exist_ok=True)
        out_path = os.path.join(output_dir, f"{pdf_name}.md")
        async with aiofiles.open(out_path, "w", encoding="utf-8") as f:
            await f.write(merged_md)
        print(f"✅ Wrote {out_path}")

    async def process_folder_sequential(self, folder_path: str, output_dir: str):
        print(f"🔍 Scanning: {folder_path}")
        pdf_files = self.get_pdf_files(folder_path)
        print(f"📚 Found {len(pdf_files)} PDFs")

        connector = aiohttp.TCPConnector(
            limit=MAX_CONCURRENT * 2,
            limit_per_host=MAX_CONCURRENT,
            ttl_dns_cache=300,
            use_dns_cache=True,
        )
        timeout = aiohttp.ClientTimeout(total=900)

        async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
            for idx, pdf in enumerate(pdf_files, 1):
                print(f"\n🚀 [{idx}/{len(pdf_files)}] Start {Path(pdf).name}")
                await self.process_single_pdf(session, pdf, output_dir)
                print(f"🏁 [{idx}/{len(pdf_files)}] Done {Path(pdf).name}")


# ------------------------------------------------------------------
# Entrypoint
# ------------------------------------------------------------------
async def main_async():
    processor = AsyncPDFProcessor(LMSTUDIO_BASE_URL, api_key=LMSTUDIO_API_KEY or "")
    await processor.process_folder_sequential(FOLDER_PATH, OUTPUT_DIR)


if __name__ == "__main__":
    await main_async()